# Turn a colleague's notebook into a reusable report

*Python Foundations · Topic 11.2*

The marketing team wants the same report as your colleague's summer sale notebook for other campaigns, starting with the July newsletter. The notebook below works, but July and the summer sale are typed into it in six places, and its last cell prints three numbers with no explanation. Turn it into a report that anyone can read, and run again by changing one cell.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Download [sessions_export_2024_07.csv](https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/sessions_export_2024_07.csv) and upload it to the `simmer-python` folder in your Drive. Choose **Runtime → Restart session and run all**, and compare the results with your colleague's text cell.
2. Add a code cell above every other code cell, with four parameters:
   - `CSV_PATH`: the file's path in your Drive
   - `START_DATE` and `END_DATE`: `"2024-07-01"` and `"2024-07-31"`
   - `CAMPAIGN_THEME`: `"summersale"`
3. Replace every hardcoded value in the cells below with a parameter. Rename `sale_rows` to `campaign_rows` and `sale_revenue` to `campaign_revenue`.
4. In the cell that reads the file, create `file_dates`: a sorted list of the dates in `rows`, each one once. Which dates are outside July?
5. Replace the last cell with one that creates `days`, the number of days from `START_DATE` to `END_DATE` with both ends counted, and `summary`, a sentence in this format, and prints it:

       From 2024-07-01 to 2024-07-31 (31 days), the summersale campaigns brought 4,023 sessions and $9,373.44 in revenue.

6. Replace your colleague's text cell with a title and a sentence that says what the report shows and how to run it for something else, with no numbers. Add a heading and a sentence above each part: the parameters, the data, the sessions by channel, the revenue and the summary.
7. Change `CAMPAIGN_THEME` to `"newsletter"`, and choose **Runtime → Restart session and run all**. The summary should say `5,563 sessions and $17,594.00 in revenue`. Change it back to `"summersale"`, and restart and run all again.

**Expected output**: after a restart and run all, the summary cell prints the sentence in step 5. The **Check your work** cell runs last, with the others. It checks the parameters, the results, that no cell below the parameters repeats them, and whether every cell in the session ran once, with no errors.

Summer sale July 2024: 4,023 sessions and $9,077.26 revenue.

In [1]:
import csv
from datetime import date

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
with open("/content/drive/MyDrive/simmer-python/sessions_export_2024_07.csv", newline="", encoding="utf-8") as file:
    rows = list(csv.DictReader(file))
len(rows)

1984

In [3]:
sale_rows = [row for row in rows if row["session_campaign"].endswith("_summersale") and "2024-07-01" <= row["date"] <= "2024-07-31"]
len(sale_rows)

135

In [4]:
channel_sessions = {}
for row in sale_rows:
    year, market, channel, theme = row["session_campaign"].split("_")
    channel_sessions[channel] = channel_sessions.get(channel, 0) + int(row["sessions"])
channel_sessions

{'facebook': 991, 'search': 1091, 'instagram': 852, 'email': 1089}

In [5]:
sale_revenue = 0
for row in sale_rows:
    sale_revenue += float(row["revenue"])
sale_revenue

9373.440000000006

In [6]:
print((date(2024, 7, 31) - date(2024, 7, 1)).days + 1, sum(channel_sessions.values()), round(sale_revenue, 2))

31 4023 9373.44


In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import os
    import re
    import sys
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    clean = False

    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            clean = True
            results.append((True, "Every cell in this session ran once, with no errors."))

    parameters = {"START_DATE": "2024-07-01", "END_DATE": "2024-07-31", "CAMPAIGN_THEME": "summersale"}
    path = found.get("CSV_PATH")
    wrong = [f"{name} is {found.get(name)!r}" for name, value in parameters.items() if found.get(name) != value]
    if not isinstance(path, str):
        results.append((False, "CSV_PATH isn't defined. Add the parameters cell above every other code cell, and run it."))
    elif not path.replace("\\", "/").endswith("sessions_export_2024_07.csv"):
        results.append((False, f"CSV_PATH is {path!r}. The report uses July's export, sessions_export_2024_07.csv."))
    elif not os.path.exists(path):
        results.append((False, f"There's no file at {path!r}. Check that Drive is mounted and that the file is in the simmer-python folder, then copy its path from the Files panel."))
    elif found.get("CAMPAIGN_THEME") == "newsletter":
        results.append((False, "CAMPAIGN_THEME is still \"newsletter\". Change it back to \"summersale\", then restart and run all."))
    elif wrong:
        results.append((False, f"{wrong[0]}. The parameters should be START_DATE = \"2024-07-01\", END_DATE = \"2024-07-31\" and CAMPAIGN_THEME = \"summersale\", with the zeros."))
    else:
        results.append((True, "The parameters are July's export, July 1 to 31 and the summer sale."))

    if history is not None:
        names = ["CSV_PATH", *parameters]
        cells = [source for source in history if re.search(r"^START_DATE\s*=", source, re.M)]
        hardcoded = ["summersale", "2024-07", "2024, 7", "sessions_export_2024_07"]
        if not cells:
            results.append((False, "No cell that has run sets START_DATE. Add the parameters cell above every other code cell."))
        else:
            parameter_cell = cells[-1]
            missing = [name for name in names if not re.search(rf"^{name}\s*=", parameter_cell, re.M)]
            below = [(value, source) for source in history if source not in cells for value in hardcoded if value in source]
            if missing:
                results.append((False, f"The parameters cell doesn't set {', '.join(missing)}. Put all four parameters in the same cell."))
            elif clean and history[0] != parameter_cell:
                results.append((False, f"The parameters cell isn't the first code cell: the first one starts with {history[0].strip().splitlines()[0]!r}. Move the parameters cell to the top."))
            elif below:
                value, source = below[0]
                results.append((False, f"{value!r} is still hardcoded in the cell that starts with {source.strip().splitlines()[0]!r}. Use the parameter instead."))
            else:
                results.append((True, "The parameters are in one cell, and no other cell repeats them."))

    rows = found.get("rows")
    file_dates = found.get("file_dates")
    if not isinstance(rows, list):
        results.append((False, "rows isn't a list. Run the cell that reads the file."))
    elif len(rows) != 1984:
        results.append((False, f"rows has {len(rows)} rows, but sessions_export_2024_07.csv has 1,984."))
    elif not isinstance(file_dates, list):
        results.append((False, "file_dates should be a sorted list of the dates in rows, each one once: sorted({row[\"date\"] for row in rows})."))
    elif len(file_dates) == 1984:
        results.append((False, "file_dates has every row's date. Put them in a set first, so each date is there once."))
    elif file_dates != sorted(file_dates):
        results.append((False, "file_dates isn't sorted. Use sorted() on the set of dates."))
    elif len(file_dates) == 33 and file_dates[-2:] == ["2024-08-01", "2024-08-02"]:
        results.append((True, "file_dates has the file's 33 dates, including August 1 and 2."))
    else:
        results.append((False, f"file_dates has {len(file_dates)} dates, but the file has 33, from 2024-07-01 to 2024-08-02."))

    campaign_rows = found.get("campaign_rows")
    if not isinstance(campaign_rows, list):
        results.append((False, "campaign_rows isn't defined. Rename sale_rows to campaign_rows, in every cell that uses it."))
    elif len(campaign_rows) == 135:
        results.append((True, "campaign_rows has the 135 summer sale rows from July 1 to 31."))
    elif len(campaign_rows) == 138:
        results.append((False, "campaign_rows has 138 rows: it includes August 1 and 2. Keep the rows where START_DATE <= row[\"date\"] <= END_DATE."))
    else:
        results.append((False, f"campaign_rows has {len(campaign_rows)} rows, but there are 135 summer sale rows from July 1 to 31. Keep the rows whose campaign ends with \"_\" + CAMPAIGN_THEME, from START_DATE to END_DATE."))

    expected = {"facebook": 991, "search": 1091, "instagram": 852, "email": 1089}
    sessions = found.get("channel_sessions")
    if sessions == expected:
        results.append((True, "channel_sessions has each channel's sessions."))
    else:
        results.append((False, f"channel_sessions should be {expected}, but it is {sessions!r}. Total the sessions of campaign_rows by channel."))

    revenue = found.get("campaign_revenue")
    if not isinstance(revenue, (int, float)):
        results.append((False, "campaign_revenue isn't defined. Rename sale_revenue to campaign_revenue, in every cell that uses it."))
    elif abs(revenue - 9373.44) < 0.01:
        results.append((True, "campaign_revenue is $9,373.44."))
    elif abs(revenue - 9077.26) < 0.01:
        results.append((False, "campaign_revenue is $9,077.26, your colleague's old number: it includes the refunds on August 1 and 2. Total the revenue of campaign_rows."))
    else:
        results.append((False, f"campaign_revenue should be about 9373.44, but it is {revenue!r}. Add float(row[\"revenue\"]) for every row in campaign_rows."))

    days = found.get("days")
    if days == 31:
        results.append((True, "days is 31."))
    elif days == 30:
        results.append((False, "days is 30: it doesn't count both ends. Add 1 to the difference's .days."))
    else:
        results.append((False, f"days should be 31, but it is {days!r}. Use (date.fromisoformat(END_DATE) - date.fromisoformat(START_DATE)).days + 1."))

    summary = found.get("summary")
    expected_summary = "From 2024-07-01 to 2024-07-31 (31 days), the summersale campaigns brought 4,023 sessions and $9,373.44 in revenue."
    if summary == expected_summary:
        results.append((True, "summary is the right sentence."))
    elif not isinstance(summary, str):
        results.append((False, "summary isn't defined. Create it in the last cell, with an f-string."))
    else:
        results.append((False, f"summary should be:\n   {expected_summary}\n   but it is:\n   {summary}\n   Use {{total_sessions:,}} and ${{campaign_revenue:,.2f}} in the f-string."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run. Change a parameter, restart and run all, and the whole report follows.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to fix it: if the check passes, your way is right too. The fixed notebook, from the top:

A text cell:

```markdown
# Campaign theme report

Sessions by channel and revenue for one campaign theme over a date range, from the store's session export. To run it for another theme or another range, change the parameters in the first code cell, then choose **Runtime → Restart session and run all**.

## Parameters
```

```python
CSV_PATH = "/content/drive/MyDrive/simmer-python/sessions_export_2024_07.csv"
START_DATE = "2024-07-01"
END_DATE = "2024-07-31"
CAMPAIGN_THEME = "summersale"
```

A text cell:

```markdown
## Data

The export, and the dates it covers. Exports can include days from the next month, so the report keeps only the rows from START_DATE to END_DATE.
```

```python
import csv
from datetime import date

from google.colab import drive

drive.mount("/content/drive")
```

```python
with open(CSV_PATH, newline="", encoding="utf-8") as file:
    rows = list(csv.DictReader(file))

file_dates = sorted({row["date"] for row in rows})
len(rows), file_dates[0], file_dates[-1]
```

```python
campaign_rows = [
    row for row in rows
    if row["session_campaign"].endswith("_" + CAMPAIGN_THEME) and START_DATE <= row["date"] <= END_DATE
]
len(campaign_rows)
```

A text cell:

```markdown
## Sessions by channel

The channel is the third part of the campaign name, as in `2024_us_email_summersale`.
```

```python
channel_sessions = {}
for row in campaign_rows:
    year, market, channel, theme = row["session_campaign"].split("_")
    channel_sessions[channel] = channel_sessions.get(channel, 0) + int(row["sessions"])
channel_sessions
```

A text cell:

```markdown
## Revenue

Total revenue of the campaign's rows, refunds included.
```

```python
campaign_revenue = 0
for row in campaign_rows:
    campaign_revenue += float(row["revenue"])
campaign_revenue
```

A text cell:

```markdown
## Summary
```

```python
days = (date.fromisoformat(END_DATE) - date.fromisoformat(START_DATE)).days + 1
total_sessions = sum(channel_sessions.values())
summary = f"From {START_DATE} to {END_DATE} ({days} days), the {CAMPAIGN_THEME} campaigns brought {total_sessions:,} sessions and ${campaign_revenue:,.2f} in revenue."
print(summary)
```

</details>